# Energizados

Step by step

### Install energizados package

In [ ]:
# !pip install energizados

### Detect Colab or Local

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"🖥️  Entorno detectado: {'Google Colab' if IN_COLAB else 'Local/Jupyter'}")

# ============================================================
# SETUP PARA COLAB: Clonar repo e instalar dependencias
# ============================================================
if IN_COLAB:
    print("📦 Configurando entorno Colab...")
    
    # Cambiar al directorio del repo (assumiendo que se montó/clonó en /content)
    import os
    if os.path.exists('/content/Energiza2Cod4Dev'):
        %cd /content/Energiza2Cod4Dev
        print("✅ Repo encontrado en /content/Energiza2Cod4Dev")
    elif os.path.exists('/content/energizados'):
        %cd /content/energizados
        print("✅ Repo encontrado en /content/energizados")
    else:
        # Clonar si no existe
        !git clone https://github.com/EL-BID/Energiza2Cod4Dev /content/Energiza2Cod4Dev
        %cd /content/Energiza2Cod4Dev
        print("✅ Repo clonado en /content/Energiza2Cod4Dev")
    
    # Instalar dependencias
    print("📥 Instalando dependencias...")
    !pip install -r requirements_colab.txt -q
    print("✅ Dependencias instaladas")
    
    # Cambiar al directorio de notebooks
    %cd /content/Energiza2Cod4Dev/notebooks
    
    # Definir paths relativos para Colab (estructura del repo clonado)
    BASE_DIR = '../'
    DATA_DIR = '../data/'
else:
    # Paths para ejecución local
    BASE_DIR = '../'
    DATA_DIR = '../data/'

print(f"📁 BASE_DIR: {BASE_DIR}")
print(f"📁 DATA_DIR: {DATA_DIR}")

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import tsfel
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from IPython.display import display, Image

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Add src to path to import energizados modules
# Nota: En el repo local es '../src', en el repo clonado para Colab es 'src'
if IN_COLAB:
    module_path = os.path.abspath(os.path.join('src'))
else:
    module_path = os.path.abspath(os.path.join('../src'))
    
if module_path not in sys.path:
    sys.path.append(module_path)
    
print(f"Module path agregado: {module_path}")

In [ ]:
from energizados.preprocessing.preprocessing import (
    fill_empty_values_str,
    fill_empty_values_cycle,
    TsfelVars,
    ExtraVars,
    ToDummy,
    TeEncoder,
    CardinalityReducer
)
from energizados.feature_selection import (
    feature_selection_by_constant,
    feature_selection_by_correlation,
    BorutaSelector
)
from energizados.modeling.simple_models import (
    ChangeTrendPercentajeIdentifierWide,
    ConstantConsumptionClassifierWide
)
from energizados.modeling.supervised_models import LGBMModel, NNModel, LSTMNNModel
from energizados.core.plots.utils import plot_roc

In [ ]:
warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:.2f}'.format
pd.set_option('display.max_columns', None)
np.set_printoptions(suppress=True)

In [ ]:
seed = 2021
np.random.seed(seed)

# Paso 1 - Leer datos
***

**Nota:** Se usa el dataset `sample_dataset.parquet` incluido en el framework. Para usar el dataset original, cambiar la ruta a `../data/df_anonimizado_02-2023.parquet`.

Descripción de las columnas:

| Variable  | Descripción | Tipo de dato | Cardinalidad |
| :--- | :--- | :--- | :--- |
| Consumo de energía mensual | Indica el comportamiento de consumo a nivel mensual de los usuarios.  Se consideran los últimos 12 consumos.| Numérica | - |
| Actividad | Indica a qué actividad económica se dedica el usuario| Categoría | 284 |
| Tipo de Tarifa | Tarifa que tipo de tarifa se le cobra al usuario| Categoría | 47 |
| Tensión | Tensión instalada al usuario.| Categoría | 18 |
| Material instalacion | Indica tipo de material del medidor instalado| Categoría | 39 |
| Zona | Indica la ubicación geográfica a la que pertenece el usuario | Categoría | 38 |
| Target | Indica si hubo un comportamiento fraudulento o no | Numérica | 0 - 1 |
| Fecha inspección | Indica la fecha en que se inspeccionó al usuario| Fecha | - |

In [ ]:
# Cargar dataset de ejemplo del framework
# Para usar el dataset original: df = pd.read_parquet(f'{DATA_DIR}df_anonimizado_02-2023.parquet')
df = pd.read_parquet(f'{DATA_DIR}sample_dataset.parquet')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
print("Proporcion de clase : ", 100*df.target.mean())

# Paso 2 - Particionar datos
***

In [ ]:
# Particionar por fecha
df_train = df[df.fecha_inspeccion<'2017-08-01'].copy()
df_val = df[(df.fecha_inspeccion>='2017-09-01')&(df.fecha_inspeccion<'2018-01-01')].copy()
df_test = df[df.fecha_inspeccion>='2018-01-01'].copy()

In [ ]:
print(df_train.shape)
print(df_val.shape)
print(df_test.shape)

In [ ]:
print("Proporcion de clase train : ", 100*df_train.target.mean())
print("Proporcion de clase validacion : ", 100*df_val.target.mean())
print("Proporcion de clase test : ", 100*df_test.target.mean())

# Paso 3 - Procesamiento de datos y construcción de modelos
***

In [ ]:
df_train.isnull().sum()

**<ins>Observación :</ins>** 

>En el conjunto de datos existen valores faltantes en las variables de consumo que son de tipos numericas y en las variables categóricas como "zona", "actividad", "tipo_tarifa" y "nivel_tension".

El tratamiento de valores faltantes fue abordado de la siguiente forma : 

- <ins>variables de consumo :</ins> se usaron los metodos ffill y bfill para propagar la observación válida hacia adelante o hacia atras.
- <ins>variables categoricas :</ins>  estas se rellenaron con una nueva categoria denominada "sin_dato".

**Nota:** En el framework actualizado, las funciones se llaman `fill_empty_values_cycle` y `fill_empty_values_str`.

In [ ]:
# Relleno de valores faltantes en serie de consumo.
# NOTE: Function names changed from llenar_val_vacios_ciclo -> fill_empty_values_cycle
df_train = fill_empty_values_cycle(df_train, 12)

# Relleno de valores faltantes en variables categoricas
# NOTE: Function names changed from llenar_val_vacios_str -> fill_empty_values_str
cols_fillna_sindatos = ['zona','actividad','tipo_tarifa','nivel_tension']
df_train = fill_empty_values_str(df_train, cols_fillna_sindatos, 'sin_dato')

In [ ]:
df_train.head()

## Modelos Simples

### Regla : Cambio o disminución en el consumo de energía

>La hipotesis detras de esta regla es que si se existen decrementos bruscos de consumos entonces es un posible comportamiento anomalo.

Configuración : 
- last_base_value : indica la cantidad de periodos anteriores para comparar.
- last_eval_value : indica la cantidad de consumos a ser evaluados.
- threshold : indica la proporción de consumo.

In [ ]:
variables_consumo = [x for x in df.columns if '_anterior' in x]
last_base_value,last_eval_value,threshold = 3,1,60
trend_perc_model = ChangeTrendPercentajeIdentifierWide(last_base_value,last_eval_value,threshold)
pred = trend_perc_model.predict(df_test[variables_consumo])

In [ ]:
# Existen un 10% de usuarios en test que cumplieron con la regla.
100*pred.is_fraud_trend_perc.value_counts(normalize=True)

In [ ]:
# usuario ejemplo que cumplio con la regla
usr = df_test.index[0]  # Ajustado para el nuevo dataset
df_test.loc[usr]

In [ ]:
plt.figure(figsize=(15,4))
y = df_test[variables_consumo].loc[usr].values
x = range(len(y))
plt.plot(x,y)
plt.scatter(x,y, color='red')
plt.ylim(0.0)
plt.grid(True)
plt.title("usr:" + str(usr)+" Cambio Trend último mes ")
plt.show()

### Regla : Consumos constante

> La hipotesis de esta regla es que si existen consumos constantes por periodos largos, entonces es un posible comportamiento anomalo.
- min_count_constante : indica la cantidad minima de periodos donde los consumo son constantes.


In [ ]:
min_count_constante =7
const_model = ConstantConsumptionClassifierWide(min_count_constante)
y_test_pred = const_model.predict(df_test[variables_consumo])

In [ ]:
# Existen aprox un 3% de usuarios en test que cumplieron con la regla.
100*y_test_pred.value_counts(normalize=True)

In [ ]:
# usuario ejemplo que cumplió con la regla
usr = df_test.index[1]  # Ajustado para el nuevo dataset
df_test.loc[usr]

In [ ]:
plt.figure(figsize=(15,4))
y = df_test[variables_consumo].loc[usr].values
x = range(len(y))
plt.plot(x,y)
plt.scatter(x,y, color='red')
plt.ylim(0.0)
plt.grid(True)
plt.title("usr:" + str(usr)+" Consumo constante ")
plt.show()

## Modelos Supervisados

### Ingenieria de variables 

> El objetivo principal es derivar variables de las serie de consumo mensual. 

**Ejemplo :** 

    1. min, maximo, pendientes.
    2. variables estadisticas,temporaales y expectrales.
    
**Paquete :**

- [TSFEL](https://tsfel.readthedocs.io/en/latest/)
- [Ejemplo de uso](https://github.com/fraunhoferportugal/tsfel/blob/master/notebooks/TSFEL_SMARTWATCH_HAR_Example.ipynb)
- Otro paquete --> [TSFRESH](https://tsfresh.readthedocs.io/en/latest/)

En el siguiente ejemplo vemos una serie de consumo, luego con el paquete TSFEL, vamos a extrar variables estadisticas que luego lo podemos usar como variables predictoras en un modelo de supervisado.

In [ ]:
serie_consumo_anteriores = [153.0,  125.0,  117.0,  120.0,  128.0,  80.0,  105.0,  123.0,  101.0,  111.0,  99.0,  96.0]
plt.figure(figsize=(10,5))
plt.plot(serie_consumo_anteriores)
plt.xticks(range(12));

In [ ]:
cfg = tsfel.get_features_by_domain("statistical")
df_result = tsfel.time_series_features_extractor(cfg, serie_consumo_anteriores,n_jobs=-1)

In [ ]:
# Como resultados tenemos una diversidad de variables estadisticas como : 0_Max	0_Mean 0_Standard deviation	0_Variance, etc.
df_result.shape

In [ ]:
df_result[['0_Skewness','0_Kurtosis', '0_Standard deviation','0_Interquartile range', '0_Kurtosis', '0_Max', '0_Mean','0_Mean absolute deviation']]

### Selección de variables 

**<ins>Nota:</ins>** En este ejemplo el proceso puede tardar más de 5 minutos! --> puede levantar las variables seleccionadas ya calculadas

> El objetivo es seleccionar las mejores variables para entrenar los modelos.

**Metodos y Paquete :**

- [Boruta](https://pypi.org/project/Boruta/)
- [Ejemplo de uso boruta](https://towardsdatascience.com/feature-selection-with-boruta-in-python-676e3877e596)
- [Mutual Information](https://towardsdatascience.com/select-features-for-machine-learning-model-with-mutual-information-534fe387d5c8)

Este paso lo realizamos luego de extraer las nuevas variables derivadas de las series de consumo. 

In [ ]:
# NOTE: Esta celda puede tardar varios minutos (TSFEL sobre 12.000 filas).
# Descomentar para ejecutar la selección de variables.
# Este paso lo vamos hacer con una muestra del conjunto de datos
variables_consumo = [x for x in df.columns if '_anterior' in x]
df_consumos = df_train[['index']+variables_consumo].head(12000)

# Construimos el pipeline de ingenieria de variables.
# TsfelVars --> Encapsula todas las funcionalidades del paquete TSFEL.
# ExtraVars --> Modulo que agrega variables extras, como cantidad de ceros seguidos en la serie de consumo y en distintas ventanas de tiempo.

pipe_feature_engeniering_consumo = Pipeline(
    [
        ("tsfel vars", TsfelVars(features_names_path=None, num_periodos=12)),
        ("add vars3",  ExtraVars(num_periodos=3)),
        ("add vars6",  ExtraVars(num_periodos=6)),
        ("add vars12", ExtraVars(num_periodos=12)),
    ]
)

df_features = pipe_feature_engeniering_consumo.fit_transform(df_consumos)

>Luego de crear nuevas variables vamos a aplicar los pasos para las seleccion de las variables mas importantes.

- Eliminamos varibles constantes
- Eliminamos las que esta altamente correlacionadas
- Seleccionamos con el metod boruta

In [ ]:
cols_for_feature_sel = [x for x in df_features.columns if x not in ['index'] + variables_consumo]
y_train = df_train.loc[df_features['index']].target

In [ ]:
%%time
select_by_constant = feature_selection_by_constant(df_features, y_train, cols_for_feature_sel, th=0.99)
print(f" # variables No constantes {len(select_by_constant)}")

select_by_corr = feature_selection_by_correlation(df_features, y_train, select_by_constant,method='pearson', th=0.95)
print(f" # variables No correlacionadas {len(select_by_corr)}")

boruta_selector = BorutaSelector(max_iter=5)
boruta_selector.fit(df_features[select_by_constant], y_train)
select_by_boruta = boruta_selector.get_selected_features()
print(f" # variables seleccionadas por Boruta : {len(select_by_boruta)}")

# Para la demo, select_by_boruta queda vacío (sin feature engineering de consumo extra)
# select_by_boruta = []  # Reemplazar por cols_for_feature_sel si se ejecutó el pipeline de arriba

In [ ]:
# Levantar variables ya seleccionadas previamente
# select_by_boruta = pd.read_csv('../data/preprocesados/features.csv')['features'].tolist()

In [ ]:
len(select_by_boruta)

### Tratamiento de las variables categoricas

> Las variables categóricas son un desafío para los algoritmos de Machine Learning. Dado que la mayoría de ellos aceptan solo valores numéricos como entradas, necesitamos transformar las categorías en números para usarlos en el modelo.

In [ ]:
variables_categoricas = ['zona','actividad','material_instalacion','tipo_tarifa','nivel_tension']

In [ ]:
df_train[variables_categoricas].head()

El tratamieno de cada variable es el siguiente : 

- __actividad__:

*Reducción de cardinalidad y dummy:* 

> Variables categóricas a las que se le redujo la cardinalidad (Esta reducción se logra, por ejemplo, agrupando valores escasos que no tienen una presencia importante en el set de datos) y luego se les aplicó One-Hot-Encoding.


- __tipo_tarifa__:

*Reducción de cardinalidad y target encoding:*

>Variables categóricas a las que se le redujo la cardinalidad y luego se las reemplazó por una medida del efecto que podrían tener en el objetivo.

- __zona y nivel_tension__:

*Variables encodeadas:*

>Variables categóricas a las que se les ha aplicado OrdinalEncoder.

- __material_instalacion__:

*Target encoding:*

>Variables categóricas a las que se le redujo la cardinalidad y luego se las reemplazó por una medida del efecto que podrían tener en el objetivo.

Nota : [Target-encoding](https://towardsdatascience.com/dealing-with-categorical-variables-by-using-target-encoder-a0f1733a4c69) 

_Finalmente el pipeline de preprocesamiento para las variables categoricas quedo configurado como se muestra a continuacion:_

```python

pipe_actividad = Pipeline([
            ('cardinality_reducer', CardinalityReducer(threshold=0.001)),
            ('a_dummy',ToDummy(['actividad']))
        ])


pipe_tarifa = Pipeline([
            ('cardinality_reducer', CardinalityReducer(threshold=0.001)),
            ('tarifa_te',TeEncoder(['tipo_tarifa'],w=20))
        ])

vars_enc = ['zona','nivel_tension']
t_features = [
    ('var_encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), vars_enc),
    ('material_isntalacion_te', TeEncoder(['material_instalacion'],w=10), ['material_instalacion']),
    ('actividad_cr_dummy', pipe_actividad, ['actividad']),
    ('tarifa_cr_te', pipe_tarifa, ['tipo_tarifa']),
    ]

preprocessor = ColumnTransformer(transformers= t_features,remainder='passthrough')

```

### Entrenamiento y evaluación de modelos supervisados

#### Procesamos los dataset de entrenamiento, validacion y test.

**<ins>Nota:</ins>** En este ejemplo este proceso puede tardar! --> puede levantar los dataset procesados

In [ ]:
y_train = df_train.target.copy()
df_train = df_train.drop(columns=['target'])

y_val = df_val.target.copy()
df_val = df_val.drop(columns=['target'])

y_test = df_test.target.copy()
df_test = df_test.drop(columns=['target'])

In [ ]:
# Realizamos los pasos de limpieza en los conjuntos de validacion y test.
df_val = fill_empty_values_cycle(df_val, 12)
df_val = fill_empty_values_str(df_val, cols_fillna_sindatos, 'sin_dato')

df_test = fill_empty_values_cycle(df_test, 12)
df_test = fill_empty_values_str(df_test, cols_fillna_sindatos, 'sin_dato')

In [ ]:
%%time
# Calculamos las variables derivadas de las series de consumo en los 3 conjuntos de datos.
df_train = pipe_feature_engeniering_consumo.fit_transform(df_train)
df_val = pipe_feature_engeniering_consumo.transform(df_val)
df_test = pipe_feature_engeniering_consumo.transform(df_test)

**<ins>Levantar datasets procesados :</ins>** 

In [ ]:
#Levantar previamente calculadas
# df_train = pd.read_parquet('../data/preprocesados/df_train_p.parquet')
# df_val = pd.read_parquet('../data/preprocesados/df_val_p.parquet')
# df_test = pd.read_parquet('../data/preprocesados/df_test_p.parquet')

In [ ]:
# Definimos las variables finales para el entrenamiento de los modelos.
feauture_selected = select_by_boruta
cols_for_model = variables_categoricas+variables_consumo+feauture_selected

In [ ]:
# Definimos el metodo de balanceo de clases con su correspondiente umbral y el pipeline de pre-procesamiento de variables categoricas.
param_imb_method = 'under'
sam_th = 0.2
periodo = 12

In [ ]:
resultado_final = {} # para guardar todas las metricas obtenidas

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

def get_preprocesor(preprocesor):
    """Build a sklearn ColumnTransformer based on a preprocessor number.

    Args:
        preprocesor: Integer selecting the preprocessing configuration.
            Currently only 4 is supported.

    Returns:
        sklearn.compose.ColumnTransformer: Configured preprocessor.
    """
    if preprocesor == 4:
        pipe_actividad = Pipeline(
            [
                ("cardinality_reducer", CardinalityReducer(threshold=0.001)),
                ("a_dummy", ToDummy(["actividad"])),
            ]
        )

        pipe_tarifa = Pipeline(
            [
                ("cardinality_reducer", CardinalityReducer(threshold=0.001)),
                ("tarifa_te", TeEncoder(["tipo_tarifa"], w=20)),
            ]
        )

        vars_enc = ["zona", "nivel_tension"]
        t_features = [
            (
                "var_encoder",
                OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
                vars_enc,
            ),
            (
                "material_isntalacion_te",
                TeEncoder(["material_instalacion"], w=10),
                ["material_instalacion"],
            ),
            ("actividad_cr_dummy", pipe_actividad, ["actividad"]),
            ("tarifa_cr_te", pipe_tarifa, ["tipo_tarifa"]),
        ]

        preprocessor = ColumnTransformer(transformers=t_features, remainder="passthrough")

    return preprocessor

#### LGBM

In [ ]:
%%time
# Importar librerías necesarias para el pipeline de LGBM
# IMPORTANTE: usar imblearn.pipeline.Pipeline (no sklearn) para soportar samplers
from imblearn.pipeline import Pipeline as ImbPipeline
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.model_selection import RandomizedSearchCV
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from scipy.stats import randint as sp_randint, uniform as sp_uniform

# Obtener el pipeline de preprocesamiento para variables categóricas
preprocessor = get_preprocesor(4)

# Construir sampler para balanceo de clases
if param_imb_method == 'under':
    sampler = RandomUnderSampler(sampling_strategy=sam_th, random_state=40)
elif param_imb_method == 'over':
    sampler = RandomOverSampler(sampling_strategy=sam_th, random_state=40)
else:
    sampler = None

# Construir clasificador LGBM
lgbm = LGBMClassifier(random_state=314, metric='None', n_estimators=1000, verbosity=-1)

# Ensamblar pipeline con imblearn.Pipeline (soporta samplers en medio del pipeline)
if sampler is not None:
    lgbm_pipeline = ImbPipeline([('preprocessor', preprocessor), ('sampler', sampler), ('lgbm', lgbm)])
else:
    lgbm_pipeline = ImbPipeline([('preprocessor', preprocessor), ('lgbm', lgbm)])

# Hiperparámetros para RandomizedSearchCV (con prefijo lgbm__)
param_test = {
    'lgbm__num_leaves': sp_randint(6, 50),
    'lgbm__max_bin': sp_randint(60, 255),
    'lgbm__max_depth': sp_randint(5, 20),
    'lgbm__min_child_samples': sp_randint(100, 500),
    'lgbm__min_child_weight': [1e-5, 1e-3, 1e-2, 1e-1, 1, 1e1, 1e2, 1e3, 1e4],
    'lgbm__subsample': sp_uniform(loc=0.2, scale=0.8),
    'lgbm__colsample_bytree': sp_uniform(loc=0.4, scale=0.6),
    'lgbm__reg_alpha': [0, 1e-1, 1, 2, 5, 7, 10, 50, 100],
    'lgbm__reg_lambda': [0, 1e-5, 1e-3, 1e-2, 1e-1, 1, 5, 10, 20, 50, 100],
    'lgbm__scale_pos_weight': [1, 5, 20, 100],
    'lgbm__learning_rate': sp_uniform(loc=0.01, scale=0.1),
    'lgbm__subsample_freq': sp_randint(5, 20),
}

# Configurar RandomizedSearchCV (sin eval_set para no filtrar val en el CV)
random_search = RandomizedSearchCV(
    estimator=lgbm_pipeline,
    param_distributions=param_test,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    n_iter=60,
    refit=True,
    random_state=314,
)
random_search.fit(df_train[cols_for_model], y_train)

best_params = random_search.best_params_
print(f"Best AUC (CV): {random_search.best_score_:.3f}")
print(f"Best params: {best_params}")

# Re-entrenar con los mejores hiperparámetros y eval_set para early stopping.
# El eval_set debe estar PREPROCESADO porque el pipeline de sklearn/imblearn no
# transforma el eval_set automáticamente — solo transforma X_train internamente.
lgbm_pipeline.set_params(**best_params)
lgbm_pipeline.fit(df_train[cols_for_model], y_train)
X_val_transformed = lgbm_pipeline.named_steps['preprocessor'].transform(df_val[cols_for_model])
lgbm_pipeline.named_steps['lgbm'].set_params(
    **{
        k.replace('lgbm__', ''): v
        for k, v in best_params.items()
    }
)
lgbm_pipeline.fit(
    df_train[cols_for_model],
    y_train,
    lgbm__eval_metric=['auc'],
    lgbm__eval_set=[(X_val_transformed, y_val)],
    lgbm__eval_names=['valid'],
    lgbm__callbacks=[
        early_stopping(stopping_rounds=30, first_metric_only=True),
        log_evaluation(0),
    ],
)

# Predecir en test
y_pred_test_lgbm = lgbm_pipeline.predict_proba(df_test[cols_for_model])[:, 1]
resulado_final[f'{param_imb_method}-lgbm'] = y_pred_test_lgbm

In [ ]:
print("AUC Test:  %.3f" % roc_auc_score(y_test, y_pred_test_lgbm))